<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 02 · Titanic Baseline

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
The chapter uses the Titanic example to show how a small supervised
learning workflow fits together: load data, define features, fit a model,
and inspect the result.

This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

module_path = BOOK_ROOT / 'code' / 'titanic_baseline.py'
spec = importlib.util.spec_from_file_location('titanic_baseline', module_path)
assert spec is not None and spec.loader is not None
titanic_baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(titanic_baseline)

print(f'Loaded {module_path.resolve()}')


This cell groups the data and orders the result so the learner can compare categories clearly.


In [ ]:
df = titanic_baseline.load_titanic_table()

print(df.head().to_string(index=False))
print()
print('weighted target counts:')
print(df.groupby('survived')['freq'].sum())
print()
print('feature counts:')
print(df.groupby(titanic_baseline.FEATURE_COLUMNS)['freq'].sum().sort_values(ascending=False).head(8))


This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


In [ ]:
X = df[titanic_baseline.FEATURE_COLUMNS]
y = df[titanic_baseline.TARGET_COLUMN]
w = df[titanic_baseline.WEIGHT_COLUMN]

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X,
    y,
    w,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

pipe = titanic_baseline.build_pipeline()
pipe.fit(X_train, y_train, model__sample_weight=w_train)
preds = pipe.predict(X_test)

print('accuracy =', round(accuracy_score(y_test, preds, sample_weight=w_test), 3))
print(confusion_matrix(y_test, preds, sample_weight=w_test))
print()
print(classification_report(y_test, preds, sample_weight=w_test, digits=3))


This cell continues the worked example for the current section.


In [ ]:
preview = X_test.copy()
preview['actual'] = y_test.values
preview['predicted'] = preds

print(preview.to_string(index=False))


### Where We Are Heading Next
Chapter 5 widens the lens to evaluation itself: metrics, confusion
matrices, and the discipline of comparing baselines fairly.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
